In [ ]:
# ===== Standard library =====
import csv
import datetime
import glob
import os
import random
import re
from collections import Counter
from datetime import timedelta

# ===== Numerical & data handling =====
import numpy as np
import pandas as pd
import polars as pl
from numpy.random import default_rng

# ===== Visualization =====
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ===== Scikit-learn =====
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit, KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# ===== Imbalanced-learn =====
from imblearn.under_sampling import RandomUnderSampler

# ===== TensorFlow / Keras =====
import tensorflow as tf
from keras.utils import to_categorical
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Activation,
    Add,
    BatchNormalization,
    Concatenate,
    Conv1D,
    Dense,
    Dropout,
    Embedding,
    Flatten,
    GlobalAveragePooling1D,
    Input,
    MaxPooling1D,
)
from tensorflow.keras.models import Model, Sequential

# ===== Project-specific modules =====
import utils.data_models as data_models
import utils.data_preprocessing as data_preprocessing
from utils.data_integration import combine_data, read_data

In [2]:
# convert all sublists to arrays
def flatten(subject):
    array_data = [np.array(x) for x in subject]  
    flattened_data = np.concatenate(array_data, axis=0)
    return flattened_data

## Read data and assing age groups

In [ ]:
"""
    Calls the "read_data()" function from file "data_integration.py"
    Calls the "combine_data()" function from file "data_integration.py"
    Parameter:
    - 1: combines only glucose data
    - 2: combines glucose and dempgraphics
    - 3: combines glucose and heartrate
    Output: This function returns the intrgrated dataset included the restricted databases if downloaded and added to folder "datasets for T1D"
 """
# reads all datasets, if they are stored in the required folder Data/datasets_for_T1D...
# DiaData includes two restricted datasets, these should be downloaded from the publishers and shored in the folder
restricted_list_5 = read_data(read_all = False, frequency = "15min")
# integrates the restricted datasets with modus = 2
df_5min = combine_data(2, restricted_list_5, columns_to_check=["Age", "Sex"])

# combine with not restricted data
# This dataset can be downloaded from https://zenodo.org/records/16875703 and should be stored in Data/datasets_for_T1D 
sdb_I_5min = pl.read_csv("Data/datasets_for_T1D/MDB.csv")
sdb_I_5min  = sdb_I_5min.sort(["PtID", "ts"])
sdb_I_5min = sdb_I_5min.to_pandas()
# converts the timestamp column to datetime
sdb_I_5min["ts"] = pd.to_datetime(sdb_I_5min["ts"])

# combines both dataframes to have the final integrated dataset 
df_5min = pd.concat([df_5min, sdb_I_5min])

In [ ]:
result_ts = df_5min.with_columns(
    pl.col("ts").str.to_datetime()
)

df_sorted = result_ts.sort(["PtID", "ts"])
df_all = df_sorted.unique(subset=["PtID", "ts"], keep="first")

n_subjects = df_all.select(pl.col("PtID").n_unique()).item()
print(n_subjects)

In [ ]:
cleaned_data = data_preprocessing.remove_outliers_polars(df_all, value="GlucoseCGM")
data_cleanedpd = cleaned_data.to_pandas()
data_cleanedpd = data_cleanedpd.sort_values(by=["PtID", "ts"]).reset_index(drop=True)

# linear interpolation is used for gaps which are less than 30 minute (6 consecuitve datapoints)
data_cleanedpd["GlucoseCGM"] = data_cleanedpd.groupby("PtID")["GlucoseCGM"].transform(
    lambda x: data_preprocessing.gap_limited_interpolation(x, limit=6)
)

# stineman interpolation for values between 30 adn 120 minutes
df_interpolated_stine = data_cleanedpd.groupby("PtID", group_keys=False).apply(lambda x: data_preprocessing.interpolate_stineman_group(x, timestamp = "ts", value = "GlucoseCGM", llimit=6, ulimit=24))
df_interpolated_stine = df_interpolated_stine.sort_index()

df_interpolated_stine = df_interpolated_stine[["ts", "GlucoseCGM", "PtID"]]
df_interpolated_stine

## features

In [ ]:
df_polars = pl.from_pandas(df_sorted)
df_polars = df_polars.sort(["PtID", "ts"])

In [ ]:
df_polars = df_polars.unique(subset=["PtID", "ts"], keep="first")

In [4]:
# Apply per SubjectID
df_features = (
    df_polars.group_by("PtID", maintain_order=True)
    .map_groups(data_preprocessing.add_time_features)) 

In [5]:
df_features = df_features.drop_nulls(["GlucoseCGM"])

In [8]:
import polars as pl
from typing import Sequence

def apply_all_metrics_lazy_per_patient(
    lf: pl.LazyFrame,
    pt_col: str = "PtID",
    ts_col: str = "ts",
    column: str = "GlucoseCGM",
    sample_every: str = "5m",
    windows: Sequence[tuple[str, str]] = (("30m", "30min"), ("1h", "1hr"), ("2h", "2hr"), ("4h", "4hr"),("8h", "8hr"),("24h", "24hr")),
) -> pl.LazyFrame:
    lf = (
        lf.with_columns(pl.col(ts_col).cast(pl.Datetime))
          .sort([pt_col, ts_col])
    )

    def to_minutes(s: str) -> int:
        s = s.strip().lower()
        if s.endswith("m"):
            return int(s[:-1])
        if s.endswith("h"):
            return int(s[:-1]) * 60
        raise ValueError(f"Unsupported duration: {s} (use 'm' or 'h')")

    samp_min = to_minutes(sample_every)
    samp_dur = pl.duration(minutes=samp_min)

    for window_size, suffix in windows:
        win_min = to_minutes(window_size)
        if win_min % samp_min != 0:
            raise ValueError(f"window_size {window_size} must be a multiple of sample_every {sample_every}")

        expected_n = win_min // samp_min
        expected_span = pl.duration(minutes=samp_min * (expected_n - 1))

        # temp column names per window
        gap_col = f"__gap_{suffix}"
        span_col = f"__span_{suffix}"
        n_col = f"__n_{suffix}"
        ok_col = f"__ok_{suffix}"

        ts = pl.col(ts_col)
        c = pl.col(column).cast(pl.Float64)

        # 1) create helper columns FIRST (no nesting window->rolling)
        lf = lf.with_columns([
            ts.diff().over(pt_col).alias(gap_col),
            (ts - ts.shift(expected_n - 1)).over(pt_col).alias(span_col),

            # count non-null samples in trailing time window
            c.is_not_null()
             .cast(pl.Int32)
             .rolling_sum_by(ts, window_size=window_size, closed="right")
             .over(pt_col)
             .alias(n_col),
        ])

        # 2) compute "window complete & gap-free" mask using helper cols
        lf = lf.with_columns([
            (
                (pl.col(n_col) == expected_n) &
                (pl.col(gap_col).rolling_max(expected_n - 1).over(pt_col) <= samp_dur) &
                (pl.col(span_col) == expected_span)
            ).alias(ok_col)
        ])

        # helper: null out if window isn't ok
        def only_if_ok(expr: pl.Expr) -> pl.Expr:
            return pl.when(pl.col(ok_col)).then(expr).otherwise(None)

        # 3) add metrics (time-based rollings) masked by ok_col
        lf = lf.with_columns([
            only_if_ok(
                c.rolling_mean_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_mean_{suffix}"),

            only_if_ok(
                c.rolling_std_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_std_{suffix}"),

            only_if_ok(
                c.rolling_min_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_min_{suffix}"),

            only_if_ok(
                c.rolling_max_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_max_{suffix}"),

            # keep a public n_{suffix} too (null when invalid)
            only_if_ok(pl.col(n_col)).alias(f"n_{suffix}"),

            only_if_ok(
                ((c < 70).cast(pl.Int8)
                 .rolling_mean_by(ts, window_size=window_size, closed="right", min_samples=expected_n)
                 .over(pt_col) * 100)
            ).alias(f"time_below70_{suffix}"),

            only_if_ok(
                ((c > 200).cast(pl.Int8)
                 .rolling_mean_by(ts, window_size=window_size, closed="right", min_samples=expected_n)
                 .over(pt_col) * 100)
            ).alias(f"time_above200_{suffix}"),

            only_if_ok((c - c.shift(expected_n - 1)).over(pt_col)).alias(f"diff_{suffix}"),
            only_if_ok(((c - c.shift(expected_n - 1)).over(pt_col)) / win_min).alias(f"slope_{suffix}"),  # per minute
        ])

        # optional: drop temp cols to keep df clean
        lf = lf.drop([gap_col, span_col, n_col, ok_col])

    return lf

In [9]:
df_features = df_features.select(['ts',
 'PtID',
 'Class1',
 'GlucoseCGM'])

In [10]:
lf_out = apply_all_metrics_lazy_per_patient(df_features.lazy())
df_out = lf_out.collect()

In [11]:
df_clean_out = df_out.drop_nulls(["GlucoseCGM_mean_30min", "GlucoseCGM_mean_1hr",
        "GlucoseCGM_mean_2hr"])
del df_out
del df_polars

In [13]:
result_ts = df_clean_out.select(['ts',
 'PtID',
 'Class1',
 'GlucoseCGM'])

## class assgnment

In [ ]:

def assign_time_before_event(df_copy, timestamp_col, class_col):

    df = df_copy.sort_values(timestamp_col).copy()

    event_times = df.loc[df[class_col] == 0, timestamp_col].to_numpy()
    timestamps  = df[timestamp_col].to_numpy()

    # include current event itself
    idx = np.searchsorted(event_times, timestamps, side="left")

    next_event_times = np.full(
        len(df),
        np.datetime64("NaT"),
        dtype="datetime64[ns]"
    )

    valid = idx < len(event_times)
    next_event_times[valid] = event_times[idx[valid]]

    df["NextEventTime"] = next_event_times

    df["TimeBfEvent"] = (
        df["NextEventTime"] - df[timestamp_col]
    )

    df["TimeBfEvent_min"] = (
        df["TimeBfEvent"].dt.total_seconds() / 60
    )

    return df

In [ ]:
df_all = result_ts.to_pandas()
df_all["ts"] = pd.to_datetime(df_all["ts"])

df_all = (
    df_all
    .sort_values(["PtID", "ts"])
    .groupby("PtID", group_keys=False)
    .apply(lambda x: assign_time_before_event(x, "ts", "Class1"))
    .reset_index(drop=True)
)
df_all = df_all.drop(columns=["NextEventTime", "TimeBfEvent"])
# sorts the glucose values by patient id and timestamps
df_all =df_all.sort_values(by=["PtID", "ts"])
# reindexes based on the sorted dataset
df_all = df_all.reset_index(drop=True)
# nneu noch mehr neu
print(df_all["Class1"].value_counts())
df_all

## extract windows

In [25]:
# Extract min and max from the scaler
x_min = df_all["GlucoseCGM"].min()
x_max = df_all["GlucoseCGM"].max()

# Compute scaled threshold
t_scaled = (70 - 40) / (500 - 40)
print("Scaled threshold:", t_scaled)

Scaled threshold: 0.06521739130434782


In [26]:
def normalize_data(df: pl.DataFrame, value: str, new_col: str) -> pl.DataFrame:
    denom = pl.col(value).max() - pl.col(value).min()
    return df.with_columns(
        pl.when(denom == 0)
        .then(0.0)
        .otherwise((pl.col(value) - pl.col(value).min()) / denom)
        .alias(new_col)
    )

In [32]:
df_imp = pl.from_pandas(df_all)
df_imp = df_imp.sort(["PtID", "ts"])

In [33]:
df_imp = normalize_data(df_imp, value="GlucoseCGM", new_col="GlucoseCGM_norm")

In [35]:
df_imp = df_imp.select(['ts',
 'PtID',
 'Class1',
 'GlucoseCGM_norm',
 'TimeBfEvent_min'])

In [39]:
df_imp

ts,PtID,Class1,GlucoseCGM_norm,TimeBfEvent_min
datetime[μs],str,i64,f64,f64
2014-10-01 21:10:00,"""001_D1NAMO""",4,0.238196,1845.0
2014-10-01 21:15:00,"""001_D1NAMO""",4,0.238196,1840.0
2014-10-01 21:20:00,"""001_D1NAMO""",4,0.238196,1835.0
2014-10-01 21:25:00,"""001_D1NAMO""",4,0.234261,1830.0
2014-10-01 21:30:00,"""001_D1NAMO""",4,0.226435,1825.0
…,…,…,…,…
2024-05-13 20:45:00,"""T1GDUJA""",4,0.128261,null
2024-05-13 20:50:00,"""T1GDUJA""",4,0.113587,null
2024-05-13 20:55:00,"""T1GDUJA""",4,0.098913,null


## time series generation

In [41]:
# generates time series with a sliding window appraoch of 2 hour lengths for the maindatabase
# splits data into train, validation, and test
def extract_valid_windows_GLC(
    df_org: pd.DataFrame,
    timestamp_col: str = "ts",
    feature_col: str = "GlucoseCGM_norm",
    class_col: str = "Class",
    expected_sample_count: int = 25,
    min_window_duration=np.timedelta64(2, "h"),
    expected_step=np.timedelta64(5, "m"),  # <-- new: required sampling period
):
    """
    This function is used genearte time series data and splits the data into train, validation, and test data.
    Parameters:
    - df_org is the original pandas dataframe,
    - timestamp_col is the name of the column with the timestamps,
    - feature_col is the name of the column which should be returned as a time series,
    - class_col is name of the column with the classes,
    - expected_sample_count is the number of allowed continuous values in a time series,
    - min_window_duration is the allowed continuous time of a time series
    - expected_step is the expected fixed time delta between consecutive samples (e.g., 5 minutes)
    Output: It returns a list including two separate lists for X_train and Y_train
    """
    # X_train, and Y_train are initialized as empty arrays

    Y_train = []
   # X_train_feat = []
    valid_ts = []
    all_ts = []


    
    # converts pandas DataFrame to polars for faster execution
    #df = pl.from_pandas(df_org)
    df =  df_org
    # sorts by timestamp to ensure chronological order
    df = df.sort(timestamp_col)
   # features = df.select(pl.exclude(["Class0","Class1", "GlucoseCGM_norm", "PtID", "ts"])).columns
    
    # extracts timestamps and total row count
    timestamps = df[timestamp_col].to_numpy()
    n_rows = df.height

    # initializes empty lists for feature windows and labels
    windows = []
    labels = []
   # windows_feature = []

    # sets starting index for window generation
    start_idx = 0

    # iterates over all rows to construct valid windows
    for end_idx in range(n_rows):
        # moves start index if window exceeds allowed duration
        while timestamps[end_idx] - timestamps[start_idx] > min_window_duration:
            start_idx += 1

        # calculates number of samples in current window
        count = end_idx - start_idx + 1

        # checks if window meets required duration and sample count
        if (
            timestamps[end_idx] - timestamps[start_idx] >= min_window_duration
            and count == expected_sample_count
        ):
            # extracts current time window
            window_df = df.slice(start_idx, count)

            # --- NEW: reject windows with gaps / irregular sampling ---
            w_ts = window_df[timestamp_col].to_numpy()
            # must be exactly expected_step between every consecutive timestamp
            if w_ts.size >= 2:
                diffs = np.diff(w_ts)
                if not np.all(diffs == expected_step):
                    continue
            # ---------------------------------------------------------

            # skips window if missing values are present
            needed_cols = [timestamp_col, feature_col, class_col]
            if window_df.select(needed_cols).null_count().sum_horizontal().item() == 0:
                # retrieves label from last row in window
                label = window_df[class_col].to_list()[-1]
                # allows only classes 0–4 (your code actually allows 0–3)
                #if label in {0, 1, 2, 3}:
                # stores feature sequence and label
                windows.append(window_df[feature_col].to_list())
                labels.append(label)

                # feat_ids = window_df.select(features).row(-1)   
                # feat_ids = [int(v) for v in feat_ids]
                # windows_feature.append(feat_ids)
                feat_ids = window_df.select("TimeBfEvent_min").row(-1)   
                #feat_ids = [int(v) for v in feat_ids]
                valid_ts.append(feat_ids)

    # checks if any valid windows were found
    if len(windows) > 0:
        # converts to numpy arrays and reshapes data for model input
        Y_data = np.array(labels).reshape(-1, 1)
       # X_data_feat = np.array(windows_feature) 

        Y_train.append(Y_data)

        X_data_feat = np.array(valid_ts) 

        all_ts.append(X_data_feat)
       # X_train_feat.append(X_data_feat)
        
        # returns feature and label arrays
        return  Y_train, all_ts 
    else:
        # returns nothing if no valid window exists
        print("No valid samples generated for this subject!")
        return None

In [42]:
def create_X_Y(df: pl.DataFrame, sample_count=25, hours=2, modus="h", class_c="Class"):
    # split into per-subject DataFrames
    df = df.sort(["PtID", "ts"])
    groups = df.partition_by("PtID", as_dict=True)

    del df

    dic_result = {}
    for ptid, df_person in groups.items():
        out = extract_valid_windows_GLC(df_person,class_col=class_c,
            expected_sample_count=sample_count,
            min_window_duration=np.timedelta64(hours, modus))
        if out is None:
            print(f"WARNING: PtID {ptid} returned None")
            continue  

        Y_data, X_ts = out
        dic_result[ptid] = {"Y": Y_data, "X_ts": X_ts}

    return dic_result

In [ ]:
import pickle, gzip

# calls the function for time series generation with a 120 min input length 
dic_data = create_X_Y(df_imp, 25, 2, 'h', class_c = "Class1")

No valid samples generated for this subject!


In [ ]:
with gzip.open("Data/misclassifcation.pkl.gz", "wb") as f:
    pickle.dump(dic_data, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
X_patients = []
Y_patients = []
subjects = []

for subj_key, payload in dic_data.items():
    subj = subj_key[0] if isinstance(subj_key, tuple) else subj_key
    chunks_X = payload["X_ts"]  
    chunks_Y = payload["Y"]  


    X_patient = np.concatenate(chunks_X, axis=0)  
    Y_patient = np.concatenate(chunks_Y, axis=0) 

    if X_patient.shape[0] != Y_patient.shape[0]:
        raise ValueError(f"{subj}: X and Y length mismatch: {X_patient.shape[0]} vs {Y_patient.shape[0]}")

    subjects.append(subj)
    X_patients.append(X_patient)
    Y_patients.append(Y_patient) 

In [ ]:
a = flatten(Y_patients)
a = flatten(a)

classes, counts = np.unique(a, return_counts=True)
classes
counts

In [ ]:
len(subjects)

In [ ]:
from sklearn.model_selection import train_test_split
# ensures reproducability
seed_value= 42 
os.environ['PYTHONHASHSEED']=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)


# assuming X_data and Y_data are lists (or arrays) where each entry corresponds to one subject
subject_indices = list(range(len(subjects)))

# split into temp and test (20%)
train_sub, test_sub = train_test_split(subjects, test_size=0.20, shuffle=True, random_state=seed_value)
train_idx, test_idx = train_test_split(subject_indices, test_size=0.20, shuffle=True, random_state=seed_value)
a= test_sub
b = test_idx
with open("test_subjects.txt", "w") as f:
    f.write("List 1:\n")
    f.write("\n".join(map(str, a)))
    f.write("\n\nList 2:\n")
    f.write("\n".join(map(str, b)))
## test with new data
from sklearn.model_selection import train_test_split
# ensures reproducability
seed_value= 42 
os.environ['PYTHONHASHSEED']=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)


# extract data by indices
X_train_all = [X_patients[i] for i in train_idx]
Y_train_all = [Y_patients[i] for i in train_idx]

X_test_all = [X_patients[i] for i in test_idx]
Y_test_all = [Y_patients[i] for i in test_idx]
print(len(X_train_all))
print(len(X_test_all))

In [ ]:
def filter_out_class_5(X_list, Y_list,  drop_value=5):
    X_filt, Y_filt = [], [] 

    for X, Y in zip(X_list, Y_list ):
        y = np.asarray(Y).reshape(-1)          # (n_i,)
        mask = y != drop_value                 # keep everything except 5

        X_filt.append(np.asarray(X)[mask])
        Y_filt.append(y[mask].reshape(-1, 1))  # keep original (n_i,1) style if you want
  

    return X_filt, Y_filt 


In [ ]:
X_train, Y_train = filter_out_class_5(X_train_all, Y_train_all,   drop_value=4)
X_test, Y_test = filter_out_class_5(X_test_all, Y_test_all,  drop_value=4)
for i, (X, Y) in enumerate(zip(X_train, Y_train)):
    assert X.shape[0] == Y.shape[0]
    assert not np.any(Y == 4)

In [ ]:
from joblib import dump, load

# Save (single file)
dump((X_test, Y_test), "Data/DL_Test_Misclass.joblib", compress=("zlib", 1))
### Prepare Data